# TAC-LAnoBERT Phase 3 Verification

**Purpose**: Runtime verification of Phase 3 implementation on Kaggle GPU.

**Steps**:
1. Setup environment (clone repo, install deps)
2. Run integration tests (8 test cases)
3. Extract timestamps for BGL dataset
4. Gradient flow test (Time2Vec params)
5. Train 2 epochs (smoke test — matches Phase 2 for fair comparison)
6. Verification report

**GPU Required**: T4 x2 or P100

**Expected Runtime**: ~10 hours (training 2 epochs on BGL 3.5M lines)

## 1. Setup Environment

In [ ]:
# Clone repository
!git clone https://github.com/rubyhcm/TAC-LAnoBERT.git
%cd TAC-LAnoBERT

# Verify structure
!ls -la tac_lanobert/

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install pytest -q

print("✓ Dependencies installed")

In [ ]:
# Verify PyTorch + CUDA
import torch
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "❌ GPU not available! Enable GPU in notebook settings."
print("\n✅ Environment ready")

## 2. Run Unit Tests & Integration Tests

Tests all TAC-LAnoBERT components:

**Unit tests** (50 tests):
- `tests/test_time2vec.py` (26): gradient flow, shape, math, normalize_delta_t
- `tests/test_memory_queue.py` (24): Welford accuracy, FIFO eviction, Mahalanobis, Ledoit-Wolf

**Integration tests** (8 tests):
- Timestamp extraction from BGL format
- Dataset loading with Time2Vec
- Model forward pass in all 4 modes (baseline, time_only, memory_only, full)
- Training step with backward pass
- Model save/load


In [ ]:
# Run all Phase 3 unit tests + integration tests
!pytest tests/test_time2vec.py tests/test_memory_queue.py tests/test_integration.py \
        -v --tb=short 2>&1 | tee pytest_output.log

# Check results
with open('pytest_output.log', 'r') as f:
    output = f.read()

if 'FAILED' in output or 'ERROR' in output:
    print("\n❌ Some tests failed! Check output above.")
    raise RuntimeError("Tests failed")
elif '58 passed' in output:
    print("\n✅ All 58 tests passed! (26 time2vec + 24 memory_queue + 8 integration)")
elif 'passed' in output:
    # Parse count
    import re
    m = re.search(r'(\d+) passed', output)
    count = int(m.group(1)) if m else '?'
    print(f"\n✅ {count} tests passed")
    if isinstance(count, int) and count < 58:
        print(f"⚠️ Expected 58 tests, got {count}. Some tests may be missing.")
else:
    print("\n⚠️ Unexpected test output, verify manually")


## 3. Extract Timestamps for BGL Dataset

Extract timestamps from BGL raw logs and save `.timestamps` sidecar files
for Time2Vec embedding during training and inference.

Output files:
- `data/BGL/BGL_train_normal_parsed.timestamps`
- `data/BGL/BGL_test_parsed.timestamps`

In [ ]:
# Verify BGL data exists, or create it if missing
import os
import glob

train_raw = "data/BGL/BGL_train_normal.raw"
test_raw  = "data/BGL/BGL_test.raw"

print("Checking BGL dataset...")
missing = False
for path in [train_raw, test_raw]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  ✓ {path}: {size_mb:.1f} MB")
    else:
        print(f"  ❌ Missing: {path}")
        missing = True

if missing:
    print("\nDataset is missing! Preparing data...")
    os.makedirs("data/BGL", exist_ok=True)
    
    print("1. Linking raw BGL log file from Kaggle dataset...")
    # Search for BGL.log anywhere in /kaggle/input/
    bgl_logs = glob.glob("/kaggle/input/**/BGL.log", recursive=True)
    if not bgl_logs:
        # Fallback search just in case
        bgl_logs = glob.glob("/kaggle/input/**/*BGL*.log", recursive=True)
        
    if bgl_logs:
        target_log = bgl_logs[0]
        print(f"  Found BGL log at: {target_log}")
        os.system(f"ln -sf {target_log} data/BGL/BGL.log")
    
    if not os.path.exists("data/BGL/BGL.log"):
        print("\n❌ Could not find BGL.log. Make sure the BGL dataset is attached to your Kaggle notebook! Click 'Add Data' and search for BGL.")
        raise FileNotFoundError("data/BGL/BGL.log")
    
    print("2. Splitting dataset (this takes a moment)...")
    !python -m tac_lanobert.split_tac --config configs/bgl_tac_full.yaml
    
    # Re-verify
    if os.path.exists(train_raw) and os.path.exists(test_raw):
        print("\n✅ Data preparation successful!")
    else:
        raise RuntimeError("Data split failed.")


In [ ]:
# Extract + preprocess training set (with timestamp extraction)
# Output: data/BGL/BGL_train_normal_parsed.log + .timestamps
print("Extracting training set timestamps...")
!python -m tac_lanobert.preprocess_tac \
    --config configs/bgl_tac_full.yaml \
    --split train \
    --extract_timestamps

# Verify .timestamps sidecar was created
ts_file = "data/BGL/BGL_train_normal_parsed.timestamps"
if os.path.exists(ts_file):
    with open(ts_file, 'r') as f:
        count = sum(1 for _ in f)
    print(f"\n✅ Train timestamps: {count:,} lines → {ts_file}")
else:
    print(f"\n❌ Timestamp file not created: {ts_file}")
    raise FileNotFoundError(ts_file)

In [ ]:
# Extract + preprocess test set (with timestamp extraction)
# Output: data/BGL/BGL_test_parsed.log + .timestamps
print("Extracting test set timestamps...")
!python -m tac_lanobert.preprocess_tac \
    --config configs/bgl_tac_full.yaml \
    --split test \
    --extract_timestamps

ts_file = "data/BGL/BGL_test_parsed.timestamps"
if os.path.exists(ts_file):
    with open(ts_file, 'r') as f:
        count = sum(1 for _ in f)
    print(f"\n✅ Test timestamps: {count:,} lines → {ts_file}")
else:
    print(f"\n❌ Timestamp file not created: {ts_file}")
    raise FileNotFoundError(ts_file)

## 3.5. Train Tokenizer

Train the WordPiece tokenizer on the parsed BGL training data.
This will create the vocab file required for model training.

In [ ]:
print("Training tokenizer...")
!python -m tac_lanobert.tokenizer_tac --config configs/bgl_tac_full.yaml

import os
vocab_file = "outputs/BGL_tac/tokenizer/BGL_LogBERT-vocab.txt"
if os.path.exists(vocab_file):
    print(f"\n✅ Tokenizer trained and saved to {vocab_file}")
else:
    print(f"\n❌ Tokenizer failed to save to {vocab_file}")
    raise FileNotFoundError(vocab_file)

## 4. Gradient Flow Test

Verify Time2Vec parameters (ω, φ) receive gradients during backprop.

In [ ]:
import torch
from transformers import BertConfig
from tac_lanobert.model import TACLAnoBERT, TACConfig

# Small config for quick test (not full 768-dim BERT)
bert_config = BertConfig(
    vocab_size=1000,
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=256,
    max_position_embeddings=128
)

tac_config = TACConfig(mode='full', num_periodic=5)
model = TACLAnoBERT(bert_config, tac_config).cuda()

print(model.get_config_summary())
print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")

# Forward + backward on small batch
batch_size, seq_len = 4, 32
input_ids    = torch.randint(0, 1000, (batch_size, seq_len)).cuda()
attention_mask = torch.ones(batch_size, seq_len).cuda()
delta_t      = torch.rand(batch_size, seq_len).cuda() * 10
labels       = input_ids.clone()

outputs = model(input_ids=input_ids, attention_mask=attention_mask, delta_t=delta_t, labels=labels)
print(f"\nLoss: {outputs.loss.item():.4f}")

outputs.loss.backward()

# Check Time2Vec parameter gradients
print("\n" + "="*60)
print("TIME2VEC GRADIENT FLOW")
print("="*60)
for name, param in model.time2vec.named_parameters():
    if param.grad is not None:
        print(f"  ✅ {name:25s}  grad_norm={param.grad.norm().item():.6f}")
    else:
        print(f"  ❌ {name:25s}  NO GRADIENT")
        raise AssertionError(f"No gradient for {name}")

print("\n✅ Gradients flowing correctly into all Time2Vec parameters!")

## 5. Train TAC-LAnoBERT (2 Epochs)

Train with `configs/bgl_tac_full.yaml`:
- 2 epochs (same as Phase 2 baseline for fair comparison)
- `gradient_accumulation_steps: 2` (effective batch = 64)
- `fp16: true`, `attn_implementation: sdpa`

**Expected runtime**: ~10h on T4 GPU

**Output**: `outputs/BGL_tac/model/final/`

In [ ]:
import time

print("="*70)
print("STARTING TRAINING — TAC-LAnoBERT, 2 EPOCHS")
print("="*70)
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")

!python -m tac_lanobert.train_tac --config configs/bgl_tac_full.yaml

print(f"\nEnd: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

In [ ]:
# Check training results
import json, os

state_file = "outputs/BGL_tac/model/trainer_state.json"
if not os.path.exists(state_file):
    import glob
    checkpoints = glob.glob("outputs/BGL_tac/model/checkpoint-*/trainer_state.json")
    if checkpoints:
        state_file = max(checkpoints, key=os.path.getmtime)

if os.path.exists(state_file):
    with open(state_file, 'r') as f:
        state = json.load(f)

    losses = [log['loss'] for log in state['log_history'] if 'loss' in log]
    eval_losses = [log['eval_loss'] for log in state['log_history'] if 'eval_loss' in log]

    print(f"Total steps: {state.get('global_step')}")
    print(f"Epoch: {state.get('epoch')}")
    print(f"First train loss: {losses[0]:.4f}")
    print(f"Final train loss: {losses[-1]:.4f}  (Δ={losses[-1]-losses[0]:+.4f})")
    if eval_losses:
        print(f"Best eval loss:   {min(eval_losses):.4f}")

    # Sanity checks
    assert all(0 < l < 50 for l in losses), "NaN/Inf detected in losses!"
    assert losses[-1] < losses[0], "Loss not decreasing!"
    print("\n✅ Loss decreasing, no NaN/Inf")
else:
    print(f"❌ trainer_state.json not found — training may have failed.")

## 6. Verification Summary

In [ ]:
import os, json
from datetime import datetime

# Check model files — transformers 4.x+ saves model.safetensors (not pytorch_model.bin)
model_dir = "outputs/BGL_tac/model/final"
model_saved = (
    os.path.exists(os.path.join(model_dir, "model.safetensors")) or
    os.path.exists(os.path.join(model_dir, "pytorch_model.bin"))
)

checks = {
    "1. Unit tests (26/26 time2vec)": os.path.exists("pytest_output.log") and
                                   "26 passed" in open("pytest_output.log").read() or
                                   ("58 passed" in open("pytest_output.log").read() if os.path.exists("pytest_output.log") else False),
    "2. Integration tests (8/8)": os.path.exists("pytest_output.log") and
                                   ("8 passed" in open("pytest_output.log").read() or
                                    "58 passed" in open("pytest_output.log").read()),
    "3. Train timestamps extracted": os.path.exists("data/BGL/BGL_train_normal_parsed.timestamps"),
    "4. Test timestamps extracted":  os.path.exists("data/BGL/BGL_test_parsed.timestamps"),
    "5. Gradient test passed":       True,  # Verified in cell above
    "6. Training completed":         model_saved,
    "7. Time2Vec weights saved":     os.path.exists(f"{model_dir}/time2vec.pt"),
    "8. TAC config saved":           os.path.exists(f"{model_dir}/tac_config.json"),
}

print("="*70)
print("PHASE 3 EXIT CRITERIA")
print("="*70)
passed = sum(1 for v in checks.values() if v)
for criterion, ok in checks.items():
    print(f"  {'✅' if ok else '❌'}  {criterion}")

print()
print(f"RESULT: {passed}/{len(checks)} checks passed")
print("="*70)

if all(checks.values()):
    print("\n🎉 PHASE 3 COMPLETE — Ready for Phase 4: Main Experiments")
else:
    failed = [k for k, v in checks.items() if not v]
    print(f"\n⚠️  Failed: {failed}")

# Save report
report = {
    "phase": 3,
    "status": "complete" if all(checks.values()) else "partial",
    "verification_date": datetime.now().isoformat(),
    "checks": checks,
    "environment": {
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    },
}

# Add training stats if available
state_file = "outputs/BGL_tac/model/trainer_state.json"
if not os.path.exists(state_file):
    import glob
    checkpoints = glob.glob("outputs/BGL_tac/model/checkpoint-*/trainer_state.json")
    if checkpoints:
        state_file = max(checkpoints, key=os.path.getmtime)

if os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)
    losses = [log['loss'] for log in state['log_history'] if 'loss' in log]
    if losses:
        report["training"] = {
            "total_steps": state.get('global_step'),
            "epoch": state.get('epoch'),
            "first_loss": losses[0],
            "final_loss": losses[-1],
            "loss_reduction": losses[0] - losses[-1],
        }

report_path = "outputs/BGL_tac/phase3_verification_report.json"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Report saved → {report_path}")
print(json.dumps(report, indent=2))

## Next Steps

After verification passes → **Phase 4: Main Experiments**:
- **E1**: Baseline Reproduction (re-confirm Phase 2 metrics)
- **E2**: Main Comparison (LAnoBERT vs TAC-LAnoBERT — F1, AUROC, FPR)
- **E3**: Early Detection (measure Detection Latency Time)
- **E4**: Ablation Study (baseline | time_only | memory_only | full)
- **E5–E7**: Robustness, Efficiency, Cross-system

See `PLAN.md` for details.